# 00 — Business Problem and PoC Design

This notebook frames the forecasting project as a business and technical PoC.

The goal is to forecast clinic usage across a healthcare network and convert those forecasts into staffing decisions. The repository is intentionally designed to show the full path from data to operational decision support.


## Problem statement

A healthcare network operates multiple clinics. Each clinic has different capacity, specialty mix, demand seasonality and marketing exposure.

The business wants to answer three practical questions:

1. How many visits should each clinic expect over the next 28 days?
2. Which clinics are likely to exceed normal capacity pressure?
3. How many clinicians, nurses and front-desk staff should be scheduled?


## Data entities

The PoC uses synthetic aggregate data with three main tables.

| Table | Grain | Description |
|---|---:|---|
| `clinic_usage` | clinic × day | visits, scheduled appointments, no-show rate, utilisation |
| `clinic_metadata` | clinic | region, size, specialty, base capacity, baseline staff |
| `marketing` | clinic × day | campaign flag and marketing spend |

No patient-level fields are generated or required.


## Modelling strategy

The PoC compares several model families:

- Seasonal naive baseline.
- Moving average baseline.
- SARIMAX with exogenous marketing variables.
- Prophet as an optional additive model.
- Global ML forecaster using lagged features and calendar variables.
- Optional LSTM and TimeGPT hooks.

This gives a balanced demonstration of statistical modelling, machine learning, deep learning and foundation-model awareness.


## Success metrics

Forecast quality is measured with MAE, RMSE, WAPE, sMAPE and bias.

Operational quality is measured by how useful the forecast is for staffing:

- fewer understaffed clinic-days,
- reduced excess staffing,
- better capacity planning after marketing campaigns,
- more stable scheduling decisions.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import SyntheticDataConfig, generate_synthetic_healthcare_data

usage, metadata, marketing = generate_synthetic_healthcare_data(
    SyntheticDataConfig(start_date="2022-01-01", end_date="2025-12-31", n_clinics=12)
)

print(usage.shape)
usage.head()


In [ ]:
summary = (
    usage.groupby("clinic_id")
    .agg(
        mean_visits=("visits", "mean"),
        p90_visits=("visits", lambda x: x.quantile(0.9)),
        mean_utilisation=("capacity_utilization", "mean"),
        max_utilisation=("capacity_utilization", "max"),
    )
    .reset_index()
)
summary.head(12)
